# G8 — Jour 1 (Personne A) : préparation de GTSDB sur Kaggle

**Avant de lancer** : à droite, *Add Input* -> ajoutez (1) le dataset GTSDB, (2) le zip du code `g8-traffic-sign-detection.zip` uploadé comme dataset (*Upload > New Dataset*). Activez *Internet* seulement si vous clonez depuis GitHub.

## 1. Où est le dataset ?

In [ ]:
!ls /kaggle/input
!find /kaggle/input -maxdepth 5 -name 'gt.txt'
!find /kaggle/input -maxdepth 5 -name '*.zip'

## 2. Récupérer le code
Choisissez UNE option. Option A : le zip uploadé comme dataset (remplacez `NOM_DATASET_CODE`).

In [ ]:
# Option A : zip uploadé comme dataset Kaggle
!find /kaggle/input -name 'g8-traffic-sign-detection.zip'
# adaptez le chemin affiché ci-dessus :
!cp /kaggle/input/NOM_DATASET_CODE/g8-traffic-sign-detection.zip /kaggle/working/
!cd /kaggle/working && unzip -q -o g8-traffic-sign-detection.zip

# Option B : GitHub (Internet ON)
# !cd /kaggle/working && git clone https://github.com/<user>/<depot>.git g8-traffic-sign-detection

In [ ]:
%cd /kaggle/working/g8-traffic-sign-detection
!pip -q install pyyaml tqdm opencv-python-headless 2>&1 | tail -1

## 3. Vérifier les annotations (`visualize_gt.py`)
Si gt.txt est dans un dossier zippé, dézippez-le d'abord dans `/kaggle/working/gtsdb_raw` puis mettez ce chemin dans `RAW`.

In [ ]:
RAW = '/kaggle/input'   # ou '/kaggle/working/gtsdb_raw' si vous avez dû dézipper
!python scripts/visualize_gt.py --raw {RAW}
from IPython.display import Image, display
display(Image('reports/gt_check.png'))

**Contrôle** : les rectangles verts doivent bien entourer les panneaux. Sinon, arrêtez-vous et envoyez-moi le message affiché.

## 4. Conversion YOLO + COCO avec split fixe

In [ ]:
!python -m src.data.convert_gtsdb --mode 4cls --raw {RAW}

## 5. Vérifications

In [ ]:
import json, glob, os
split = json.load(open('splits/split.json'))
print({k: len(v) for k, v in split.items()})   # attendu : ~480 / ~120 / 300
base = 'data/processed/gtsdb_4cls'
for sp in ['train', 'val', 'test']:
    n_img = len(glob.glob(f'{base}/images/{sp}/*.png'))
    n_lbl = sum(1 for f in glob.glob(f'{base}/labels/{sp}/*.txt') if os.path.getsize(f) > 0)
    coco = json.load(open(f'{base}/coco/{sp}.json'))
    print(sp, '| images:', n_img, '| images avec panneaux:', n_lbl, '| boîtes COCO:', len(coco['annotations']))
print(open(f'{base}/dataset.yaml').read())

## 6. Sauvegarder
- Téléchargez `splits/split.json` (panneau *Output*, ou la cellule ci-dessous) et **committez-le** dans le dépôt.
- Cliquez sur **Save Version > Save & Run All (Commit)** : les données converties deviennent un *Output* réutilisable comme *Input* dans vos notebooks d'entraînement.

In [ ]:
!cp splits/split.json /kaggle/working/split.json
!du -sh data/processed